**CELL 9**  Read CSV into Spark DataFrame


In [ ]:
from pyspark.sql import functions as F

df_raw = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .option("multiLine", "true")
    .option("escape", "\"")
    .csv(CSV_PATH)
)

print("Rows:", df_raw.count())
print("Cols:", len(df_raw.columns))
df_raw.printSchema()
df_raw.show(5, truncate=False)

Rows: 5986025
Cols: 19
root
 |-- ARREST_KEY: integer (nullable = true)
 |-- ARREST_DATE: string (nullable = true)
 |-- PD_CD: integer (nullable = true)
 |-- PD_DESC: string (nullable = true)
 |-- KY_CD: integer (nullable = true)
 |-- OFNS_DESC: string (nullable = true)
 |-- LAW_CODE: string (nullable = true)
 |-- LAW_CAT_CD: string (nullable = true)
 |-- ARREST_BORO: string (nullable = true)
 |-- ARREST_PRECINCT: integer (nullable = true)
 |-- JURISDICTION_CODE: integer (nullable = true)
 |-- AGE_GROUP: string (nullable = true)
 |-- PERP_SEX: string (nullable = true)
 |-- PERP_RACE: string (nullable = true)
 |-- X_COORD_CD: double (nullable = true)
 |-- Y_COORD_CD: double (nullable = true)
 |-- Latitude: double (nullable = true)
 |-- Longitude: double (nullable = true)
 |-- Lon_Lat: string (nullable = true)

+----------+-----------+-----+-----------------+-----+--------------+----------+----------+-----------+---------------+-----------------+---------+--------+--------------+---------

**CELL 10**  Standardize column names



In [ ]:
import re

def clean_colname(c: str) -> str:
    c = c.strip()
    c = re.sub(r"\s+", "_", c)
    c = re.sub(r"[^0-9a-zA-Z_]", "", c)
    return c

df = df_raw.toDF(*[clean_colname(c) for c in df_raw.columns])
print(df.columns)

['ARREST_KEY', 'ARREST_DATE', 'PD_CD', 'PD_DESC', 'KY_CD', 'OFNS_DESC', 'LAW_CODE', 'LAW_CAT_CD', 'ARREST_BORO', 'ARREST_PRECINCT', 'JURISDICTION_CODE', 'AGE_GROUP', 'PERP_SEX', 'PERP_RACE', 'X_COORD_CD', 'Y_COORD_CD', 'Latitude', 'Longitude', 'Lon_Lat']


**CELL 11**  Convert empty strings to NULL + quick null snapshot



In [ ]:
for c in df.columns:
    df = df.withColumn(
        c,
        F.when(F.trim(F.col(c).cast("string")) == "", F.lit(None)).otherwise(F.col(c))
    )

# quick null count snapshot
df.select([F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in df.columns]).show(truncate=False)

+----------+-----------+-----+-------+-----+---------+--------+----------+-----------+---------------+-----------------+---------+--------+---------+----------+----------+--------+---------+-------+
|ARREST_KEY|ARREST_DATE|PD_CD|PD_DESC|KY_CD|OFNS_DESC|LAW_CODE|LAW_CAT_CD|ARREST_BORO|ARREST_PRECINCT|JURISDICTION_CODE|AGE_GROUP|PERP_SEX|PERP_RACE|X_COORD_CD|Y_COORD_CD|Latitude|Longitude|Lon_Lat|
+----------+-----------+-----+-------+-----+---------+--------+----------+-----------+---------------+-----------------+---------+--------+---------+----------+----------+--------+---------+-------+
|0         |0          |884  |9169   |9788 |9169     |196     |24990     |8          |0              |10               |17       |0       |0        |1         |1         |5       |5        |5      |
+----------+-----------+-----+-------+-----+---------+--------+----------+-----------+---------------+-----------------+---------+--------+---------+----------+----------+--------+---------+-------+



**CELL 12** Parse date + enforce numeric types



In [ ]:
# Date parsing (NYPD arrest data usually supports to_date directly)
if "ARREST_DATE" in df.columns:
    df = df.withColumn("ARREST_DATE", F.to_date(F.col("ARREST_DATE")))

# Cast numeric fields safely
for c in ["PD_CD","KY_CD","ARREST_PRECINCT","JURISDICTION_CODE"]:
    if c in df.columns:
        df = df.withColumn(c, F.col(c).cast("int"))

for c in ["X_COORD_CD","Y_COORD_CD","Latitude","Longitude"]:
    if c in df.columns:
        df = df.withColumn(c, F.col(c).cast("double"))

df.printSchema()

root
 |-- ARREST_KEY: integer (nullable = true)
 |-- ARREST_DATE: date (nullable = true)
 |-- PD_CD: integer (nullable = true)
 |-- PD_DESC: string (nullable = true)
 |-- KY_CD: integer (nullable = true)
 |-- OFNS_DESC: string (nullable = true)
 |-- LAW_CODE: string (nullable = true)
 |-- LAW_CAT_CD: string (nullable = true)
 |-- ARREST_BORO: string (nullable = true)
 |-- ARREST_PRECINCT: integer (nullable = true)
 |-- JURISDICTION_CODE: integer (nullable = true)
 |-- AGE_GROUP: string (nullable = true)
 |-- PERP_SEX: string (nullable = true)
 |-- PERP_RACE: string (nullable = true)
 |-- X_COORD_CD: double (nullable = true)
 |-- Y_COORD_CD: double (nullable = true)
 |-- Latitude: double (nullable = true)
 |-- Longitude: double (nullable = true)
 |-- Lon_Lat: string (nullable = true)



**CELL 13** Remove duplicates


In [ ]:
before = df.count()

if "ARREST_KEY" in df.columns:
    df = df.dropDuplicates(["ARREST_KEY"])
else:
    df = df.dropDuplicates()

after = df.count()
print("Before:", before, "| After:", after, "| Removed:", before - after)

Before: 5986025 | After: 5986025 | Removed: 0


**3) Feature Engineering for a Clear Target**

**CELL 14** Create the target label (SEVERITY) from LAW_CAT_CD



In [ ]:
# LAW_CAT_CD is typically: F (Felony), M (Misdemeanor), V (Violation)
if "LAW_CAT_CD" in df.columns:
    df = df.withColumn(
        "SEVERITY",
        F.when(F.upper(F.col("LAW_CAT_CD")).isin("F", "FELONY"), "FELONY")
         .when(F.upper(F.col("LAW_CAT_CD")).isin("M", "MISDEMEANOR"), "MISDEMEANOR")
         .when(F.upper(F.col("LAW_CAT_CD")).isin("V", "VIOLATION"), "VIOLATION")
         .otherwise("OTHER")
    )

df.select([c for c in ["LAW_CAT_CD","SEVERITY"] if c in df.columns]).show(20, truncate=False)

+----------+-----------+
|LAW_CAT_CD|SEVERITY   |
+----------+-----------+
|F         |FELONY     |
|M         |MISDEMEANOR|
|I         |OTHER      |
|F         |FELONY     |
|M         |MISDEMEANOR|
|M         |MISDEMEANOR|
|M         |MISDEMEANOR|
|F         |FELONY     |
|M         |MISDEMEANOR|
|M         |MISDEMEANOR|
|F         |FELONY     |
|M         |MISDEMEANOR|
|M         |MISDEMEANOR|
|M         |MISDEMEANOR|
|M         |MISDEMEANOR|
|M         |MISDEMEANOR|
|M         |MISDEMEANOR|
|F         |FELONY     |
|F         |FELONY     |
|F         |FELONY     |
+----------+-----------+
only showing top 20 rows



**CELL 15** Coordinate sanity filtering



In [ ]:
if "Latitude" in df.columns and "Longitude" in df.columns:
    df = df.filter(
        (F.col("Latitude").isNull() | ((F.col("Latitude") >= -90) & (F.col("Latitude") <= 90))) &
        (F.col("Longitude").isNull() | ((F.col("Longitude") >= -180) & (F.col("Longitude") <= 180)))
    )

print("Rows after coord filter:", df.count())

Rows after coord filter: 5986025


**4) Tableau-Ready Clean Export**

**CELL 16** Write cleaned dataset to a single CSV folder



In [ ]:
OUT_DIR = "/content/NYPD_Tableau_Cleaned"
(
    df.coalesce(1)
      .write
      .mode("overwrite")
      .option("header", "true")
      .csv(OUT_DIR)
)

!ls -lh {OUT_DIR}

total 1.3G
-rw-r--r-- 1 root root 1.3G Feb  9 19:11 part-00000-3268e452-c19a-4ce9-b2b0-ab93e1129c3e-c000.csv
-rw-r--r-- 1 root root    0 Feb  9 19:11 _SUCCESS


**CELL 17** Copy Spark part file into a single named CSV



In [ ]:
import glob, shutil

part_files = glob.glob(f"{OUT_DIR}/part-*.csv")
print("Part files:", part_files)

FINAL_CSV = "/content/NYPD_Tableau_Cleaned.csv"
shutil.copy(part_files[0], FINAL_CSV)

!ls -lh /content | grep NYPD_Tableau_Cleaned.csv

Part files: ['/content/NYPD_Tableau_Cleaned/part-00000-3268e452-c19a-4ce9-b2b0-ab93e1129c3e-c000.csv']
-rw-r--r-- 1 root root 1.3G Feb  9 19:12 NYPD_Tableau_Cleaned.csv


**CELL 18** Download cleaned CSV



In [ ]:
from google.colab import files
files.download(FINAL_CSV)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>